# Ray + MLflow Smoke Run 指南

这个 Notebook 只负责交互式检查、提交和查看结果。正式训练仍由 `scripts/train.py` 执行，数据、模型、训练与 MLflow 写入逻辑全部位于 `src/ray_cats_dogs/`。

默认执行不会创建 MLflow Experiment/Run，也不会启动训练。只有显式设置 `RUN_SMOKE = True` 才会通过 Ray Jobs API 提交一个可恢复的 Smoke Job。

In [ ]:
CONFIG_NAME = "smoke.yaml"
RUN_SMOKE = False
FORCE_NEW_ATTEMPT = False
RAY_JOBS_ADDRESS = "http://127.0.0.1:8265"
EXISTING_SUBMISSION_ID = None
FOLLOW_JOB = True
POLL_INTERVAL_SECONDS = 5

## 1. 定位项目并加载正式配置

无论 JupyterLab 从仓库根目录还是项目目录启动，下面的单元都会定位 `ray-cats-and-dogs`。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        candidates = (base, base / "train-model" / "ray-cats-and-dogs")
        for candidate in candidates:
            if (candidate / "configs" / "smoke.yaml").is_file():
                return candidate
    raise FileNotFoundError("Cannot locate train-model/ray-cats-and-dogs")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIG_PATH = PROJECT_ROOT / "configs" / CONFIG_NAME

from ray_cats_dogs.config import load_config
from ray_cats_dogs.runtime import build_runtime_env
from ray_cats_dogs.train import config_plan, read_only_plan

config = load_config(CONFIG_PATH)
print(f"project_root={PROJECT_ROOT}")
print(f"config={CONFIG_PATH}")

## 2. 平台只读检查

检查 MinIO、MLflow 和 Ray，不修改服务状态。

In [ ]:
checks = {
    "services": ["systemctl", "is-active", "minio.service", "mlflow.service"],
    "mlflow_health": ["curl", "-fsS", "-H", "Host: localhost", "http://127.0.0.1:5000/health"],
    "minio_health": ["curl", "-fsS", "http://127.0.0.1:9000/minio/health/live"],
    "ray_status": ["ray", "status"],
}
platform_status = {}
for name, command in checks.items():
    completed = subprocess.run(command, check=False, capture_output=True, text=True)
    platform_status[name] = {
        "returncode": completed.returncode,
        "output": (completed.stdout or completed.stderr).strip(),
    }
print(json.dumps(platform_status, indent=2, ensure_ascii=False))

## 3. 配置与资源检查

这里只做本地配置校验，不访问数据或 MLflow。主目标必须是验证指标，Smoke 不允许读取测试集。

In [ ]:
resolved_plan = config_plan(config)
print(json.dumps(resolved_plan, indent=2, ensure_ascii=False, sort_keys=True))

## 4. 全量只读计划

该步骤验证全部图片、生成被 Git 忽略的确定性 Manifest、读取 MLflow Tracking API 并计算幂等键，但不会创建 Experiment/Run 或启动 Worker。

In [ ]:
training_plan = read_only_plan(config)
print(json.dumps(training_plan, indent=2, ensure_ascii=False, sort_keys=True))

## 5. 显式提交 Smoke Ray Job

确认前面检查无误后，将参数单元的 `RUN_SMOKE` 改成 `True` 并重新执行本单元。Job 使用正式脚本和 `smoke.yaml`，Notebook 不承载训练进程。

In [ ]:
submission_id = EXISTING_SUBMISSION_ID
if RUN_SMOKE:
    if config.run.role != "smoke":
        raise ValueError("Notebook submission requires a smoke-role config")
    from ray.job_submission import JobSubmissionClient

    jobs_client = JobSubmissionClient(RAY_JOBS_ADDRESS)
    entrypoint = f"python scripts/train.py --config configs/{CONFIG_NAME}"
    if FORCE_NEW_ATTEMPT:
        entrypoint += " --force"
    submission_id = jobs_client.submit_job(
        entrypoint=entrypoint,
        runtime_env=build_runtime_env(PROJECT_ROOT),
        metadata={"project": config.project_name, "role": config.run.role},
        entrypoint_num_cpus=1,
        entrypoint_memory=512 * 1024**2,
    )
    print(f"submitted_ray_job={submission_id}")
else:
    print("RUN_SMOKE=False; no Ray Job was submitted.")

## 6. 查看 Ray Job 状态与日志

提交后可重复执行。也可以把已有 Job ID 填入参数单元的 `EXISTING_SUBMISSION_ID`。`FOLLOW_JOB=True` 时会每隔几秒轮询 Ray Job，从日志展示训练/验证 batch 进度条，并从 MLflow Tracking API 展示 epoch 总进度和历史指标；设为 `False` 可只查看一次。

In [ ]:
import time
import pandas as pd
from IPython.display import HTML, clear_output, display
from mlflow import MlflowClient
from ray.job_submission import JobSubmissionClient

tracking_client = MlflowClient(tracking_uri=config.mlflow.tracking_uri)
TERMINAL_JOB_STATUSES = {"SUCCEEDED", "FAILED", "STOPPED", "STOPPING"}
PROGRESS_METRICS = (
    "train_loss", "train_accuracy", "train_precision", "train_recall",
    "train_f1", "train_macro_f1", "val_loss", "val_accuracy",
    "val_precision", "val_recall",
    "val_f1", "val_macro_f1",
    "best_objective", "epoch_duration_seconds", "train_examples_per_second",
    "val_examples_per_second",
)

def _status_name(status):
    return str(getattr(status, "value", status)).rsplit(".", 1)[-1].upper()


def _json_event(line):
    start = line.find("{")
    end = line.rfind("}")
    if start < 0 or end < start:
        return {}
    try:
        return json.loads(line[start:end + 1])
    except json.JSONDecodeError:
        return {}


def _run_id_from_logs(logs):
    for line in logs.splitlines():
        event = _json_event(line)
        if event.get("event") == "run-started":
            return event.get("mlflow_run_id")
    return None


def _metrics_from_mlflow(run_id):
    if not run_id:
        return pd.DataFrame()
    rows = {}
    try:
        for metric_name in PROGRESS_METRICS:
            for metric in tracking_client.get_metric_history(run_id, metric_name):
                if metric.step <= 0:
                    continue
                rows.setdefault(int(metric.step), {})[metric_name] = metric.value
    except Exception:
        return pd.DataFrame()
    if not rows:
        return pd.DataFrame()
    return (
        pd.DataFrame.from_dict(rows, orient="index")
        .rename_axis("epoch")
        .reset_index()
        .sort_values("epoch")
    )


def _metrics_from_logs(logs):
    rows = {}
    for line in logs.splitlines():
        event = _json_event(line)
        if event.get("event") == "epoch-complete":
            epoch = int(event["epoch"])
            rows[epoch] = {
                name: event[name] for name in PROGRESS_METRICS if name in event
            }
    if not rows:
        return pd.DataFrame()
    return (
        pd.DataFrame.from_dict(rows, orient="index")
        .rename_axis("epoch")
        .reset_index()
    )


def _render_progress(status_name, logs):
    clear_output(wait=True)
    print(f"ray_job={submission_id} status={status_name}")
    run_id = _run_id_from_logs(logs)
    progress = _metrics_from_mlflow(run_id)
    if progress.empty:
        progress = _metrics_from_logs(logs)
    if progress.empty:
        print("等待 Ray Worker 上报第一个 epoch...")
    else:
        completed_epoch = int(progress["epoch"].max())
        total_epochs = int(config.training.epochs)
        display(HTML(
            f"<progress value='{completed_epoch}' max='{total_epochs}'></progress> "
            f"epoch {completed_epoch}/{total_epochs}"
        ))
        display(progress)
    if logs:
        print("\n最近日志:")
        print("\n".join(logs.splitlines()[-12:]))


if submission_id:
    jobs_client = JobSubmissionClient(RAY_JOBS_ADDRESS)
    while True:
        job_status = jobs_client.get_job_status(submission_id)
        status_name = _status_name(job_status)
        job_logs = jobs_client.get_job_logs(submission_id)
        _render_progress(status_name, job_logs)
        if not FOLLOW_JOB or status_name in TERMINAL_JOB_STATUSES:
            break
        time.sleep(POLL_INTERVAL_SECONDS)
else:
    print("No submission ID is selected.")

## 7. 通过 MLflow API 查看 Smoke Runs

只读取 Tracking API，不访问 `mlflow.db` 或 MinIO 服务端文件系统。

In [ ]:
import pandas as pd
from mlflow import MlflowClient

tracking_client = MlflowClient(tracking_uri=config.mlflow.tracking_uri)
experiment = tracking_client.get_experiment_by_name(config.mlflow.experiment_name)
if experiment is None:
    print("Experiment does not exist yet; submit the first Smoke Run when ready.")
else:
    runs = tracking_client.search_runs(
        [experiment.experiment_id],
        filter_string="tags.`project` = 'ray-cats-and-dogs' and tags.`run.role` = 'smoke'",
        order_by=["attributes.start_time DESC"],
        max_results=20,
    )
    rows = [
        {
            "run_id": run.info.run_id,
            "status": run.info.status,
            "outcome": run.data.tags.get("run.outcome"),
            "best_val_accuracy": run.data.metrics.get("best_objective"),
            "last_val_loss": run.data.metrics.get("val_loss"),
            "last_val_precision": run.data.metrics.get("val_precision"),
            "last_val_recall": run.data.metrics.get("val_recall"),
            "last_val_f1": run.data.metrics.get("val_f1"),
            "last_train_accuracy": run.data.metrics.get("train_accuracy"),
            "last_epoch_seconds": run.data.metrics.get("epoch_duration_seconds"),
            "artifact_verified": run.data.tags.get("artifact.roundtrip_verified"),
            "ray_job_id": run.data.tags.get("ray.job_id"),
        }
        for run in runs
    ]
    display(pd.DataFrame(rows))

## 完成条件

Smoke Run 应满足：Ray Job 为 `SUCCEEDED`、MLflow Run 为 `FINISHED`、`run.outcome=succeeded`、`artifact.roundtrip_verified=true`，并且 `test.evaluated=false`。通过后再提交验证集 Trial；不要直接运行 Champion 或修改 Registry Alias。